In [0]:
%pip install openml

In [0]:
import os
import pandas as pd
from sklearn.datasets import fetch_openml

# 1. Crear carpeta data/raw si no existe
os.makedirs("data/raw", exist_ok=True)

# 2. Descargar Frecuencia
print("Descargando freMTPL2freq desde OpenML...")
freq = fetch_openml(data_id=41214, as_frame=True, parser="auto").frame
freq.to_csv("data/raw/freMTPL2freq.csv", index=False)
print("-> freMTPL2freq.csv guardado en data/raw/")

# 3. Descargar Severidad
print("Descargando freMTPL2sev desde OpenML...")
sev = fetch_openml(data_id=41215, as_frame=True, parser="auto").frame
sev.to_csv("data/raw/freMTPL2sev.csv", index=False)
print("-> freMTPL2sev.csv guardado en data/raw/")

print("\n¡Descarga finalizada con éxito!")

In [0]:
import pandas as pd

# Leer los archivos recién creados
df_freq = pd.read_csv("data/raw/freMTPL2freq.csv")
df_sev = pd.read_csv("data/raw/freMTPL2sev.csv")

# Imprimir dimensiones
print("=== BASE DE FRECUENCIA ===")
print(f"Filas (Pólizas): {df_freq.shape[0]:,}")
print(f"Columnas: {df_freq.shape[1]}")

print("\n=== BASE DE SEVERIDAD ===")
print(f"Filas (Siniestros): {df_sev.shape[0]:,}")
print(f"Columnas: {df_sev.shape[1]}")

In [0]:
# 1. Información general y nulos en Frecuencia
print("=== NULOS EN FRECUENCIA ===")
print(df_freq.isnull().sum())
print("\n=== TIPOS DE DATOS EN FRECUENCIA ===")
print(df_freq.dtypes)

# 2. Información general y nulos en Severidad
print("\n=== NULOS EN SEVERIDAD ===")
print(df_sev.isnull().sum())

In [0]:
# Reglas de validación actuarial
exp_invalid = df_freq[df_freq["Exposure"] <= 0]
exp_over_1 = df_freq[df_freq["Exposure"] > 1]
claim_neg = df_freq[df_freq["ClaimNb"] < 0]

print("=== AUDITORÍA DE CONSISTENCIA ===")
print(f"Pólizas con Exposición <= 0: {len(exp_invalid)}")
print(f"Pólizas con Exposición > 1: {len(exp_over_1)}")
print(f"Pólizas con ClaimNb < 0: {len(claim_neg)}")

# Resumen estadístico de Exposure y ClaimNb
print("\n=== RESUMEN ESTADÍSTICO DE EXPOSICIÓN Y SINIESTROS ===")
df_freq[["Exposure", "ClaimNb"]].describe()

In [0]:
# Siniestros con monto <= 0
sev_zero_or_less = df_sev[df_sev["ClaimAmount"] <= 0]

print("=== AUDITORÍA DE SEVERIDAD ===")
print(f"Siniestros con ClaimAmount <= 0: {len(sev_zero_or_less)}")

print("\n=== RESUMEN ESTADÍSTICO DE MONTO DE SINIESTROS ===")
df_sev["ClaimAmount"].describe()

In [0]:
# 1. Copia de resguardo
df_freq_clean = df_freq.copy()
df_sev_clean = df_sev.copy()

# 2. Capping de Exposición (máximo 1.0)
df_freq_clean["Exposure"] = df_freq_clean["Exposure"].clip(upper=1.0)

# 3. Capping de Frecuencia (máximo 4 siniestros)
df_freq_clean["ClaimNb"] = df_freq_clean["ClaimNb"].clip(upper=4)

# 4. Capping de Severidad a €10.000 para control de atípicos extremos
SEVERITY_CAP = 10000
df_sev_clean["ClaimAmount_Capped"] = df_sev_clean["ClaimAmount"].clip(upper=SEVERITY_CAP)

# 5. Verificación de resultados tras la limpieza
print("=== VERIFICACIÓN TRAS LIMPIEZA ===")
print(f"Max Exposure corregido: {df_freq_clean['Exposure'].max()}")
print(f"Max ClaimNb corregido: {df_freq_clean['ClaimNb'].max()}")
print(f"Siniestros por encima del cap (€{SEVERITY_CAP:,}): {(df_sev['ClaimAmount'] > SEVERITY_CAP).sum()} de {len(df_sev)}")
print(f"Max ClaimAmount Capped: €{df_sev_clean['ClaimAmount_Capped'].max():,}")

In [0]:
import numpy as np

# 1. Agrupamiento por Area para calcular Frecuencia Observada
area_stats = df_freq_clean.groupby("Area").agg(
    Pólizas=("IDpol", "count"),
    Exposicion_Total=("Exposure", "sum"),
    Siniestros_Totales=("ClaimNb", "sum")
).reset_index()

area_stats["Frecuencia_Observada"] = area_stats["Siniestros_Totales"] / area_stats["Exposicion_Total"]

print("=== FRECUENCIA OBSERVADA POR ÁREA GEOGRÁFICA ===")
print(area_stats.sort_values(by="Frecuencia_Observada", ascending=False))

# 2. Guardar las bases limpias en la carpeta data/processed
import os
os.makedirs("data/processed", exist_ok=True)

df_freq_clean.to_csv("data/processed/freq_clean.csv", index=False)
df_sev_clean.to_csv("data/processed/sev_clean.csv", index=False)

print("\n¡Bases procesadas guardadas exitosamente en data/processed/!")